In [3]:
import os
import random
import shutil

# --- Configuration ---
base_dir = "data_class_10"
classes = ["hairline", "medium", "large"]

# Target directories
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

# Split ratios
train_ratio = 0.8
val_ratio   = 0.1
test_ratio  = 0.1

valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')

# --- Create the target folders ---
for d in [train_dir, val_dir, test_dir]:
    for c in classes:
        os.makedirs(os.path.join(d, c), exist_ok=True)

print("Starting dataset splitting (80% Train, 10% Val, 10% Test)...\n")

for class_name in classes:
    source_folder = os.path.join(base_dir, class_name)
    
    if not os.path.exists(source_folder):
        print(f"❌ Warning: Source folder '{source_folder}' does not exist. Skipping.")
        continue
        
    # Get all valid images and shuffle them for random distribution
    files = [f for f in os.listdir(source_folder) if f.lower().endswith(valid_exts)]
    random.shuffle(files)
    
    total_files = len(files)
    if total_files == 0:
        print(f"❌ Warning: No images found in '{source_folder}'. Skipping.")
        continue
        
    # Calculate exact split indices
    train_end = int(total_files * train_ratio)
    val_end   = train_end + int(total_files * val_ratio)
    
    train_files = files[:train_end]
    val_files   = files[train_end:val_end]
    test_files  = files[val_end:]
    
    print(f"📁 Class '{class_name}' (Total: {total_files} images)")
    print(f"   -> Train : {len(train_files)} images")
    print(f"   -> Val   : {len(val_files)} images")
    print(f"   -> Test  : {len(test_files)} images")
    
    # Copy files to their respective directories
    for f in train_files:
        shutil.copy2(os.path.join(source_folder, f), os.path.join(train_dir, class_name, f))
    for f in val_files:
        shutil.copy2(os.path.join(source_folder, f), os.path.join(val_dir, class_name, f))
    for f in test_files:
        shutil.copy2(os.path.join(source_folder, f), os.path.join(test_dir, class_name, f))

print("\n✅ Dataset splitting complete! You are now ready to run the offline augmentation on the train folder.")

Starting dataset splitting (80% Train, 10% Val, 10% Test)...

❌ Warning: Source folder 'data_class_10\hairline' does not exist. Skipping.
❌ Warning: Source folder 'data_class_10\medium' does not exist. Skipping.
❌ Warning: Source folder 'data_class_10\large' does not exist. Skipping.

✅ Dataset splitting complete! You are now ready to run the offline augmentation on the train folder.


In [4]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

base_dir = "data_class_10"
train_dir = os.path.join(base_dir, "train")

offline_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

def augment_to_target(train_class_dir, target_count=10000):
    valid_exts = ('.jpg', '.png', '.jpeg', '.bmp')
    current_images = [f for f in os.listdir(train_class_dir) if f.lower().endswith(valid_exts)]
    current_count = len(current_images)

    if current_count >= target_count:
        print(f"Skipping {os.path.basename(train_class_dir)}: Already has {current_count} images.")
        return

    print(f"Augmenting {os.path.basename(train_class_dir)}: From {current_count} to {target_count} images...")
    
    idx = 0
    while current_count < target_count:
        img_name = current_images[idx % len(current_images)]
        img_path = os.path.join(train_class_dir, img_name)
        
        try:
            img = load_img(img_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)

            for batch in offline_datagen.flow(x, batch_size=1,
                                              save_to_dir=train_class_dir,
                                              save_prefix="aug",
                                              save_format="jpg"):
                current_count += 1
                if current_count >= target_count:
                    break
        except Exception as e:
            pass
            
        idx += 1

# Apply offline augmentation to each training subfolder
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        augment_to_target(class_path, target_count=10000)

# Verify the counts
print("\n--- VERIFICATION ---")
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.png','.jpeg','.bmp'))])
        print(f"{class_name}: {count} images")


KeyboardInterrupt



In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import class_weight

# --- 1. CRITICAL FIX: Use 128x128 or higher for crack detection ---
# 64x64 destroys the fine details of hairline/medium cracks.
IMG_SIZE = (224, 224) 
BATCH_SIZE = 64

base_dir = "data_class_10"
train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

# --- 2. Add Online Data Augmentation to combat overfitting ---
# Since we are removing EarlyStopping, the model WILL overfit without this.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation and Test sets should NOT be augmented, only rescaled
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')
test_gen = val_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

# --- 3. Calculate Class Weights to fix the "Medium" class ---
# The model is ignoring "medium" because it's harder to learn. This forces it to pay attention.
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights_array))
print(f"Class weights applied: {class_weights_dict}")

# --- 4. Rebuild the Model (Slightly stronger regularization) ---
tf.keras.backend.clear_session()

def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_custom_crack_cnn(input_shape=(128, 128, 3), num_classes=3):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = residual_block(x, 64);  x = layers.MaxPooling2D((2, 2))(x)
    x = residual_block(x, 128); x = layers.MaxPooling2D((2, 2))(x)
    x = residual_block(x, 256); x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    # Increased Dropout from 0.4 to 0.5 to fight overfitting
    x = layers.Dropout(0.5)(x) 
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return models.Model(inputs, outputs, name="Custom_ResNet_Crack_Classifier")

model = build_custom_crack_cnn(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=3)

# --- 5. Compile with a slightly lower learning rate for stability ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# --- 6. Callbacks (EarlyStopping REMOVED as requested) ---
cb = [
    # Keep ReduceLROnPlateau so it lowers LR if it gets stuck, but no early stopping.
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1, min_lr=1e-6)
]

# --- 7. Train for more epochs (since we removed EarlyStopping) ---
# Because we added online data augmentation, it will take longer to converge.
history = model.fit(
    train_gen,
    epochs=30, # Increased epochs
    validation_data=val_gen,
    class_weight=class_weights_dict, # Apply class weights
    callbacks=cb
)

In [ ]:
# Save model
model.save("multiclass_concrete_crack.h5")

In [ ]:
import tensorflow as tf
import os

# Define the filename for the converted model
tflite_model_file = 'crack_classifier_model.tflite'

print("Converting model to TensorFlow Lite format...")

# 1. Create the TFLite Converter from your trained Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. Apply Optimizations (Optional but highly recommended)
# This reduces the model size by ~4x and speeds up inference, with minimal accuracy loss.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. Convert the model
try:
    tflite_model = converter.convert()
    
    # 4. Save the .tflite file to disk
    with open(tflite_model_file, 'wb') as f:
    #with open(multiclass_concrete_crack, 'wb') as f:
        f.write(tflite_model)
        
    model_size_mb = os.path.getsize(tflite_model_file) / (1024 * 1024)
    print(f"✅ Success! Model converted and saved as '{tflite_model_file}'")
    print(f"📦 TFLite Model Size: {model_size_mb:.2f} MB")

except Exception as e:
    print(f"❌ Conversion failed. Error: {e}")

In [ ]:
model.summary()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, 
    accuracy_score, precision_score, recall_score, f1_score
)

# ==========================================
# GLOBAL SETTINGS FOR EXTREME CLARITY
# ==========================================
plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 24,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'figure.titlesize': 26
})

# ==========================================
# 1A. TRAINING ACCURACY PLOT (Separate)
# ==========================================
plt.figure(figsize=(10, 7))
plt.plot(history.history['accuracy'], 'b-', linewidth=3, label='Train Accuracy')
plt.plot(history.history['val_accuracy'], 'r-', linewidth=3, label='Validation Accuracy')
plt.title('Training and Validation Accuracy', pad=15)
plt.xlabel('Epochs', labelpad=10)
plt.ylabel('Accuracy', labelpad=10)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
# Save as .tif with 600 DPI
plt.savefig('training_accuracy.tif', dpi=600, bbox_inches='tight')
plt.show()

# ==========================================
# 1B. TRAINING LOSS PLOT (Separate)
# ==========================================
plt.figure(figsize=(10, 7))
plt.plot(history.history['loss'], 'b-', linewidth=3, label='Train Loss')
plt.plot(history.history['val_loss'], 'r-', linewidth=3, label='Validation Loss')
plt.title('Training and Validation Loss', pad=15)
plt.xlabel('Epochs', labelpad=10)
plt.ylabel('Loss', labelpad=10)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
# Save as .tif with 600 DPI
plt.savefig('training_loss.tif', dpi=600, bbox_inches='tight')
plt.show()

# ==========================================
# 2. GENERATE PREDICTIONS
# ==========================================
print("Generating predictions on the Test Set...")
y_pred_probs = model.predict(test_gen)
y_true = test_gen.classes
y_pred_classes = np.argmax(y_pred_probs, axis=1)
class_names = list(test_gen.class_indices.keys())
n_classes = len(class_names)

# ==========================================
# 3. PRINT FINAL METRICS
# ==========================================
acc = accuracy_score(y_true, y_pred_classes)
prec = precision_score(y_true, y_pred_classes, average='macro')
rec = recall_score(y_true, y_pred_classes, average='macro')
f1 = f1_score(y_true, y_pred_classes, average='macro')

print("\n" + "="*50)
print("=== FINAL TEST METRICS ===")
print(f"Accuracy  : {acc * 100:.2f}%")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("="*50)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# ==========================================
# 4. CONFUSION MATRIX
# ==========================================
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            annot_kws={"size": 22, "weight": "bold"})
plt.title('Confusion Matrix', pad=20)
plt.xlabel('Predicted Class', labelpad=15)
plt.ylabel('True Class', labelpad=15)
plt.tight_layout()
# Save as .tif with 600 DPI
plt.savefig('confusion_matrix.tif', dpi=600, bbox_inches='tight')
plt.show()

# ==========================================
# 5. MULTICLASS ROC / AUC CURVE
# ==========================================
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=n_classes)

fpr, tpr, roc_auc = {}, {}, {}
plt.figure(figsize=(12, 10))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_onehot[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=4, 
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=3, label='Random Guess')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.title('Multiclass ROC / AUC Curve', pad=20)
plt.xlabel('False Positive Rate', labelpad=15)
plt.ylabel('True Positive Rate', labelpad=15)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
# Save as .tif with 600 DPI
plt.savefig('roc_auc_curve.tif', dpi=600, bbox_inches='tight')
plt.show()